# Consuming Machine Learning Models from Applications

In the last notebook you deployed a Titanic prediction API to the cloud. That was the hard part. This notebook is about the **last mile**: how real software — websites, mobile apps, backend jobs, even spreadsheets — actually *uses* a deployed model. The good news: every one of them does it the exact same way. They send an HTTP request with JSON, and they get JSON back. Once you can do that well from Python, you can do it from anywhere. This is the final notebook of the course, so we finish by looking back at everything you built.

**What you will learn:**

- The vocabulary of API consumption: endpoint, request, response, payload, status codes
- How to call a prediction API from Python with the `requests` library
- The professional habits of a good API client: timeouts, error handling, retries
- How to write a reusable client function that looks like real production code
- Single calls vs batch calls, and why fewer round trips means faster
- How a plain web page calls the same API with JavaScript `fetch()`
- How the API can serve its own web page (same origin, relative `fetch`), so the app runs locally and inside Colab
- How third-party ML APIs (OpenAI, Anthropic, Google Vision) fit the same pattern — and where API keys must live
- Production concerns for consumers: latency, caching, graceful degradation, versioning

## How to run this notebook

- **Locally:** open it in Jupyter (`jupyter lab` or `jupyter notebook`) and run cells top to bottom.
- **Google Colab:** upload the notebook (or open it from GitHub) and run cells top to bottom.

The setup cell below installs any missing packages, so both environments work out of the box. **No accounts and no API keys are needed** — everything runs on the machine executing the notebook.

One special thing about this notebook: it starts a **real (local) web server inside the notebook** and then talks to it over **real HTTP**. The requests you send are genuine network calls, exactly like the ones your app would send to a cloud API. This is the closest thing to production you can do in a classroom.

In [1]:
# SETUP - run this cell first. It installs any missing packages.
# On Google Colab most packages are already installed, so this usually does nothing.
import importlib.util, subprocess, sys

def ensure(package, pip_name=None):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or package])

ensure("pandas"); ensure("numpy"); ensure("sklearn", "scikit-learn"); ensure("fastapi"); ensure("uvicorn"); ensure("requests")

## 1. The last mile: a deployed model is just a URL

After deployment, your model is a **URL** — an address on the internet (or on your own machine). Nothing more. Every kind of application consumes it the same way:

- a **website** calls the URL from JavaScript
- a **mobile app** calls the URL from Swift or Kotlin
- a **backend job** calls the URL from Python or Java
- even a **spreadsheet** can call the URL

They all do exactly one thing: **send an HTTP request with JSON in, get JSON back.** Learn that once and you can consume any model from anything.

### The vocabulary (one line each)

- **Endpoint** — one specific URL path the API answers on, like `/health` or `/predict`.
- **Request** — the message your app sends to the endpoint.
- **Response** — the message the API sends back.
- **Payload** — the data inside the request (for us: a JSON object describing a passenger).

### GET vs POST: read vs send-data

- **GET** means *"read something"*. No payload needed. Example: `GET /health` asks "are you alive?".
- **POST** means *"here is data, do something with it"*. The payload travels in the **body** of the request. Example: `POST /predict` sends a passenger and asks for a prediction.

### Status codes: the API's one-number summary

Every response comes with a **status code** — a three-digit number that summarizes how the call went. These are the ones you will actually meet, each with a Titanic-API example:

| Code | Meaning | When you would see it with our API |
|---|---|---|
| 200 | OK — everything worked | You POST a valid passenger and get a prediction back |
| 400 | Bad request — your input made no sense | You send a body that is not even valid JSON |
| 404 | Not found — wrong URL | You call `/predictt` (typo) instead of `/predict` |
| 422 | Validation failed — right shape, wrong content | You send `"Age": "twenty"` — text where a number belongs |
| 500 | Server broke — a bug on *their* side | The model code crashes while predicting |
| 503 | Unavailable — server temporarily overloaded or restarting | Cloud Run is starting a new instance and cannot answer yet |

Rule of thumb: **2xx = success, 4xx = your fault, 5xx = their fault.** We will trigger several of these on purpose later — reading error responses is a core skill, not a failure.

## 2. Build and start the model API

We need a model API to consume. We rebuild the exact API from the previous notebook (`02-deploying-models-in-the-cloud`) in compact form — go back there for the full explanations of FastAPI, Pydantic, and deployment. Today the API is just our test subject; consuming it is the topic.

**What the next cell does / why:** it downloads the Titanic dataset, applies the same feature preparation as always (Title from Name, FamilySize, IsAlone), and trains a random forest inside a pipeline. One small note: **no train/test split today** — evaluating models was M2's job. Today the model just needs to exist so we have something to call.

In [2]:
# Compact Titanic prep + model training (full explanation: M2 and M3 notebook 2)
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Feature engineering (canonical prep used throughout the course)
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df.loc[~df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Title"] = "Rare"
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

numeric = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
categorical = ["Pclass", "Sex", "Embarked", "Title"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

model = Pipeline([
    ("prep", preprocess),
    ("forest", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)),
])
model.fit(df[numeric + categorical], df["Survived"])
print("Model trained on", len(df), "passengers")

Model trained on 891 passengers


**What the next cell does / why:** it defines the FastAPI app — the same three ideas as notebook 2:

- `GET /health` answers `{"status": "ok"}` so anyone can check the API is alive.
- `POST /predict` takes one passenger and returns a prediction plus a probability.
- `POST /predict-batch` takes a **list** of passengers and predicts them all in one call. We add it now (routes must exist before the server starts) but we will only *use* it in section 5.

Two details worth noting:

- **Pydantic model** (`Passenger`): FastAPI uses it to validate incoming JSON automatically. Wrong types get rejected with a 422 before our code even runs.
- **CORSMiddleware** with `allow_origins=["*"]`: this tells browsers "any web page may call me". Fine for a class project; real apps list only their own website. We explain CORS properly in section 6.

In [3]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Titanic Survival API")

# CORS: lets web pages in the browser call this API (see section 6).
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class Passenger(BaseModel):
    Pclass: int
    Sex: str
    Age: float
    SibSp: int
    Parch: int
    Fare: float
    Embarked: str

def passenger_to_frame(passengers):
    """Turn Passenger objects into the DataFrame our pipeline expects."""
    rows = []
    for p in passengers:
        d = p.model_dump()
        # The API asks only for raw fields; we derive the engineered ones here.
        d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
        if d["Sex"] == "male":
            d["Title"] = "Master" if d["Age"] < 13 else "Mr"
        else:
            d["Title"] = "Miss" if d["Age"] < 18 else "Mrs"
        rows.append(d)
    return pd.DataFrame(rows)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(passenger: Passenger):
    X = passenger_to_frame([passenger])
    return {
        "prediction": int(model.predict(X)[0]),
        "probability": round(float(model.predict_proba(X)[0][1]), 4),
    }

@app.post("/predict-batch")
def predict_batch(passengers: list[Passenger]):
    X = passenger_to_frame(passengers)
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {"predictions": [
        {"prediction": int(p), "probability": round(float(pr), 4)}
        for p, pr in zip(preds, probs)
    ]}

print("API defined with routes:", [r.path for r in app.routes if not r.path.startswith(("/openapi", "/docs", "/redoc"))])

API defined with routes: ['/health', '/predict', '/predict-batch']


**What the next cell does / why:** it starts a real `uvicorn` web server on `127.0.0.1:8123` in a **background thread** (a second line of execution inside the same program), so the notebook can keep running cells while the server listens. The loop below waits until the server actually answers before we move on.

> **Important:** `127.0.0.1` (also called *localhost*) means "this same machine". This localhost URL behaves **exactly** like your Cloud Run URL from notebook 2. To point all of today's code at your real cloud API, change **one variable**: `BASE_URL`. Everything else stays identical — that is the whole point of APIs.

In [4]:
import threading, time, uvicorn, requests

BASE_URL = "http://127.0.0.1:8123"   # swap for your Cloud Run URL and everything below still works

config = uvicorn.Config(app, host="127.0.0.1", port=8123, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

# wait until the server answers (max ~15 s)
for _ in range(60):
    try:
        requests.get(f"{BASE_URL}/health", timeout=0.5)
        break
    except requests.exceptions.RequestException:
        time.sleep(0.25)
print("Server is running at", BASE_URL)

Server is running at http://127.0.0.1:8123


## 3. Consuming from Python with `requests`

`requests` is Python's most popular library for making HTTP calls. Every API call — no matter the language — has the same **four ingredients**:

| Ingredient | What it is | Our example |
|---|---|---|
| URL | where to send it | `http://127.0.0.1:8123/predict` |
| Method | what kind of action | `GET` (read) or `POST` (send data) |
| Payload | the data you send (POST only) | a JSON object describing a passenger |
| Headers | small labels about the request | `Content-Type: application/json` (requests sets this for us) |

**What the next cell does / why:** the simplest possible call — `GET /health`. We look at the three things every response gives you: the status code, and the body as JSON.

In [5]:
response = requests.get(f"{BASE_URL}/health")

print("Status code:", response.status_code)   # 200 means OK
print("JSON body:  ", response.json())        # .json() turns the JSON text into a Python dict

Status code: 200
JSON body:   {'status': 'ok'}


**What the next cell does / why:** the real thing — `POST /predict` with a passenger. With `requests.post`, the `json=` argument does two jobs at once: it converts our Python dict to JSON text and sets the right header. We pretty-print the response so it is easy to read.

In [6]:
import json

passenger = {
    "Pclass": 1, "Sex": "female", "Age": 24,
    "SibSp": 0, "Parch": 0, "Fare": 80.0, "Embarked": "C",
}

response = requests.post(f"{BASE_URL}/predict", json=passenger)
result = response.json()

print("Status code:", response.status_code)
print(json.dumps(result, indent=2))   # pretty-print the JSON response

Status code: 200
{
  "prediction": 1,
  "probability": 0.995
}


**What the next cell does / why:** the API gave us raw numbers: a `prediction` (0 or 1) and a `probability`. Numbers are not a user experience. **Your app decides how to SHOW the prediction — the API only provides it.** The same API response can become a sentence, a color, a chart, or a push notification. Here we turn it into a friendly sentence.

In [7]:
prob = result["probability"]
verdict = "would likely have survived" if result["prediction"] == 1 else "would likely not have survived"
print(f"This passenger {verdict} the Titanic (survival probability: {prob:.1%}).")

This passenger would likely have survived the Titanic (survival probability: 99.5%).


## 4. Being a good client

Anyone can send one request on a sunny day. Professionals write clients that behave well when things go wrong — because on the internet, things go wrong constantly: servers restart, networks hiccup, someone sends bad data. Four habits separate toy code from production code: **timeouts, error handling, retries, and wrapping it all in one reusable function.** All four are below, all runnable.

### 4a. Timeouts: never wait forever

A **timeout** is the maximum time you are willing to wait for an answer. Without one, `requests` will happily wait *forever* if the server hangs — and your whole app hangs with it. Rule: **always pass `timeout=`.**

**What the next cell does / why:** the same health call, now with a 5-second timeout. The server answers in milliseconds, so nothing visible changes — but if the server ever froze, this call would raise an error after 5 seconds instead of hanging your app forever.

In [8]:
response = requests.get(f"{BASE_URL}/health", timeout=5)
print("Answered with status", response.status_code, "well within the 5-second limit")

Answered with status 200 well within the 5-second limit


### 4b. Error handling: read what the API tells you

`response.raise_for_status()` is a one-liner that raises a Python exception whenever the status code is 4xx or 5xx. Put it inside `try/except` and no bad response ever slips through silently.

**What the next cell does / why:** we misbehave *on purpose*, twice:

1. We POST `"Age": "twenty"` — text where a number belongs. The API rejects it with **422** and, crucially, the error body **tells you exactly what is wrong**. Reading the error body is how you debug API calls.
2. We call `/predictt` (a typo) and handle the **404** gracefully instead of crashing.

In [9]:
# Mistake 1: invalid input -> 422, and the API explains the problem
bad_passenger = {**passenger, "Age": "twenty"}   # Age must be a number!
response = requests.post(f"{BASE_URL}/predict", json=bad_passenger, timeout=5)
try:
    response.raise_for_status()
except requests.exceptions.HTTPError:
    print("Got status", response.status_code, "- the API tells us what is wrong:")
    error = response.json()["detail"][0]
    print("  field:  ", error["loc"][-1])
    print("  problem:", error["msg"])

# Mistake 2: wrong URL -> 404, handled gracefully
response = requests.get(f"{BASE_URL}/predictt", timeout=5)   # typo on purpose
try:
    response.raise_for_status()
except requests.exceptions.HTTPError:
    print("\nGot status", response.status_code, "- that endpoint does not exist. Check the URL.")

Got status 422 - the API tells us what is wrong:
  field:   Age
  problem: Input should be a valid number, unable to parse string as a number

Got status 404 - that endpoint does not exist. Check the URL.


### 4c. Retries with backoff: try again, but politely

Networks hiccup. Cloud servers restart. Sometimes a request fails for a reason that fixes itself in two seconds. The professional answer is a **retry**: try again after a short pause, and make each pause longer than the last (**backoff**), so you do not hammer a struggling server.

This is just a for-loop with a `sleep` — a loop pattern you already know from M1.

**When retrying makes sense and when it does not:**

- **Retry:** 503 (server temporarily unavailable), network hiccups, timeouts. The problem may fix itself.
- **Do NOT retry:** 400 or 422 — *your data is wrong*. Sending the same wrong data five times gives you the same error five times.

**What the next cell does / why:** a small retry function with increasing sleeps. We show both faces: it succeeds instantly against our healthy server, and it retries then gives up cleanly against a port where nothing is listening.

In [10]:
def get_with_retries(url, attempts=3, timeout=5):
    """GET a URL; on connection problems retry with increasing pauses."""
    for attempt in range(1, attempts + 1):
        try:
            return requests.get(url, timeout=timeout)
        except requests.exceptions.RequestException as err:
            print(f"  attempt {attempt} failed: {type(err).__name__}")
            if attempt == attempts:
                return None                 # out of attempts: give up cleanly
            time.sleep(0.3 * attempt)       # backoff: 0.3 s, then 0.6 s, ...
    return None

print("Healthy server:")
response = get_with_retries(f"{BASE_URL}/health")
print("  success on first try:", response.json())

print("Dead server (port 9999, nothing listening there):")
response = get_with_retries("http://127.0.0.1:9999/health", timeout=1)
print("  final result:", response)   # None - we gave up politely instead of crashing

Healthy server:
  success on first try: {'status': 'ok'}
Dead server (port 9999, nothing listening there):
  attempt 1 failed: ConnectionError


  attempt 2 failed: ConnectionError


  attempt 3 failed: ConnectionError
  final result: None


### 4d. Putting it together: a reusable client function

**What the next cell does / why:** it combines all the habits — timeout, retries with backoff, status checking, clean error reporting — into one function: `get_prediction(passenger)`. **This ~15-line function is what production client code actually looks like.** Real apps hide all API details behind a function like this; the rest of the codebase just calls it.

Then we use it like an application would: loop over 5 different passengers, collect the results into a DataFrame, display.

In [11]:
def get_prediction(passenger: dict, attempts: int = 3) -> dict:
    """Call the prediction API like a professional: timeout, retries, error handling."""
    for attempt in range(1, attempts + 1):
        try:
            response = requests.post(f"{BASE_URL}/predict", json=passenger, timeout=5)
            if response.status_code in (400, 422):          # our data is wrong: retrying will not help
                return {"error": f"invalid input ({response.status_code}): {response.json()['detail']}"}
            response.raise_for_status()                     # any other 4xx/5xx raises here
            return response.json()                          # success
        except requests.exceptions.RequestException as err:
            if attempt == attempts:
                return {"error": f"gave up after {attempts} attempts: {type(err).__name__}"}
            time.sleep(0.3 * attempt)                       # backoff before the next try

passengers = [
    {"Pclass": 1, "Sex": "female", "Age": 24, "SibSp": 0, "Parch": 0, "Fare": 80.0, "Embarked": "C"},
    {"Pclass": 3, "Sex": "male",   "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.9,  "Embarked": "S"},
    {"Pclass": 2, "Sex": "female", "Age": 35, "SibSp": 1, "Parch": 2, "Fare": 26.0, "Embarked": "S"},
    {"Pclass": 3, "Sex": "male",   "Age": 4,  "SibSp": 3, "Parch": 1, "Fare": 21.1, "Embarked": "Q"},
    {"Pclass": 1, "Sex": "male",   "Age": 60, "SibSp": 1, "Parch": 0, "Fare": 120.0, "Embarked": "C"},
]

results = pd.DataFrame([{**p, **get_prediction(p)} for p in passengers])
results

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,prediction,probability
0,1,female,24,0,0,80.0,C,1,0.9950
1,3,male,28,0,0,7.9,S,0,0.0936
2,2,female,35,1,2,26.0,S,1,0.9140
3,3,male,4,3,1,21.1,Q,0,0.0911
4,1,male,60,1,0,120.0,C,0,0.3163


## 5. Batch consumption: many predictions at once

Suppose your app needs predictions for **many** rows — a nightly job scoring every customer, for example. The naive approach is a loop of single calls. It works, but every call pays the **round trip**: request travels to the server, server answers, response travels back. Do that 10 times and you pay the travel cost 10 times.

That is why real APIs often accept a **list**: one call, many predictions. One round trip, and the model can process all rows together (which is also faster for the model itself). Our API has had `/predict-batch` since section 2 — time to use it.

**What the next cell does / why:** it predicts 10 passengers both ways — a loop of 10 single calls vs one batch call — and measures the elapsed time of each with `time.perf_counter()` (a high-precision stopwatch).

In [12]:
ten_passengers = (passengers * 2)[:10]   # reuse our 5 passengers twice = 10 rows

# Way 1: a loop of 10 single calls (10 round trips)
start = time.perf_counter()
loop_results = [requests.post(f"{BASE_URL}/predict", json=p, timeout=5).json() for p in ten_passengers]
loop_seconds = time.perf_counter() - start

# Way 2: one batch call (1 round trip)
start = time.perf_counter()
batch_results = requests.post(f"{BASE_URL}/predict-batch", json=ten_passengers, timeout=5).json()["predictions"]
batch_seconds = time.perf_counter() - start

print(f"Loop of 10 single calls: {loop_seconds*1000:6.1f} ms")
print(f"One batch call:          {batch_seconds*1000:6.1f} ms")
print(f"Speedup: {loop_seconds/batch_seconds:.1f}x  (same {len(batch_results)} predictions)")
print("Identical results:", loop_results == batch_results)

Loop of 10 single calls:   97.9 ms
One batch call:             9.9 ms
Speedup: 9.9x  (same 10 predictions)
Identical results: True


Fewer round trips = faster. On localhost a round trip costs a millisecond or two; to a real cloud server it costs tens to hundreds of milliseconds, so the batch advantage gets *much* bigger in production.

## 6. Consuming from a web page

Time to consume the API the way most humans meet ML: through a **web page**. A page needs three things: **HTML** (structure: the form), **CSS** (looks), and **JavaScript** (behavior: calling the API). The JavaScript function for HTTP calls is called `fetch()` — and it mirrors `requests.post` one-to-one.

**What the next cell does / why:** it writes a complete, minimal web page to `outputs/webapp/index.html`: a form (class, sex, age, fare), about 20 lines of plain CSS, and a `fetch()` call — no frameworks, nothing to install.

In [13]:
from pathlib import Path

webapp_dir = Path("outputs") / "webapp"
webapp_dir.mkdir(parents=True, exist_ok=True)

html_page = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Titanic Survival Predictor</title>
<style>
  body { font-family: system-ui, sans-serif; background: #f4f5f7; margin: 0; }
  main { max-width: 420px; margin: 3rem auto; background: white; padding: 2rem;
         border-radius: 8px; box-shadow: 0 1px 4px rgba(0,0,0,0.12); }
  h1 { font-size: 1.3rem; margin-top: 0; }
  label { display: block; margin-top: 0.8rem; font-size: 0.9rem; color: #333; }
  select, input { width: 100%; padding: 0.5rem; margin-top: 0.25rem;
                  border: 1px solid #ccc; border-radius: 4px; box-sizing: border-box; }
  button { margin-top: 1.2rem; width: 100%; padding: 0.7rem; border: none;
           border-radius: 4px; background: #1a5fb4; color: white;
           font-size: 1rem; cursor: pointer; }
  button:hover { background: #14487f; }
  #result { margin-top: 1.2rem; font-size: 1.05rem; min-height: 1.5rem; }
</style>
</head>
<body>
<main>
  <h1>Titanic Survival Predictor</h1>
  <label>Passenger class
    <select id="pclass"><option value="1">1st</option><option value="2">2nd</option>
    <option value="3" selected>3rd</option></select>
  </label>
  <label>Sex
    <select id="sex"><option value="female">female</option>
    <option value="male" selected>male</option></select>
  </label>
  <label>Age <input id="age" type="number" value="28" min="0" max="100"></label>
  <label>Fare (pounds) <input id="fare" type="number" value="15" min="0" step="0.1"></label>
  <button onclick="predict()">Predict survival</button>
  <p id="result"></p>
</main>
<script>
const BASE_URL = "http://127.0.0.1:8123";  // change to your Cloud Run URL to go live

async function predict() {
  const passenger = {
    Pclass: Number(document.getElementById("pclass").value),
    Sex: document.getElementById("sex").value,
    Age: Number(document.getElementById("age").value),
    SibSp: 0, Parch: 0,
    Fare: Number(document.getElementById("fare").value),
    Embarked: "S",
  };
  const out = document.getElementById("result");
  out.textContent = "Asking the model...";
  try {
    const response = await fetch(BASE_URL + "/predict", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify(passenger),
    });
    const result = await response.json();
    const pct = Math.round(result.probability * 100);
    out.textContent = "Survival probability: " + pct + "%";
  } catch (err) {
    out.textContent = "Could not reach the model API. Is it running?";
  }
}
</script>
</body>
</html>
"""

page_path = webapp_dir / "index.html"
page_path.write_text(html_page, encoding="utf-8")
print("Web page written next to this notebook:", page_path)

Web page written next to this notebook: outputs/webapp/index.html


### The `fetch()` call, line by line

It mirrors `requests.post` one-to-one:

```javascript
const response = await fetch(BASE_URL + "/predict", {   // the URL - same as in Python
  method: "POST",                                       // the method - like requests.post
  headers: { "Content-Type": "application/json" },      // "I am sending JSON" (requests did this for us)
  body: JSON.stringify(passenger),                      // dict -> JSON text (json= did this for us)
});
const result = await response.json();                    // parse the response - same as .json() in Python
```

- `await` means "wait for the answer without freezing the page" — the browser's polite way of waiting.
- Same four ingredients: URL, method, payload, headers. Only the spelling changed.

### CORS in plain words

**CORS** (Cross-Origin Resource Sharing) is a **browser safety rule: a web page may only call other servers if that server says it's OK.** Your page (opened from a file, or served from some website) is one "origin"; the API at `127.0.0.1:8123` is another. Before the browser lets the page talk to the API, it checks whether the API allows it. That is exactly why our API has `CORSMiddleware`: with `allow_origins=["*"]` it says "any page may call me". Fine for a class project — a real app would list only its own website, e.g. `allow_origins=["https://myapp.com"]`. Note: this rule only exists inside browsers; Python `requests` is not affected.

### How to try the page

- **Locally:** while this notebook is running (server still up), find `outputs/webapp/index.html` next to this notebook and double-click it. It opens in your browser and works end to end: form, real HTTP call, live prediction.
- **On Colab:** the server lives inside *Google's* machine, so *your* browser cannot reach its `127.0.0.1` — the downloaded file will not work as-is. You could edit the page's `BASE_URL` to your Cloud Run URL from notebook 2 (that works from anywhere on earth) — but the cells below show a neater trick that makes the app work **inside Colab** with zero changes.

### Bonus: serve the page from the API itself — and make it work on Colab

Right now the page is a separate file, and the file must know the API's address (`BASE_URL`) — that is what forced us to think about CORS. There is a neater pattern, used by many real products: **let the API serve its own web page.** The page and the API then share one address (one "origin"), so:

- the JavaScript can call just `fetch("/predict")` — a **relative URL**, no address configured anywhere;
- no CORS needed — same origin, so the browser has no objection;
- wherever the API runs (your laptop, Colab, Cloud Run), the page automatically calls the right server — because it came *from* that server.

**What the next cell does / why:** it adds a `GET /` route to our API that returns the same page, with `BASE_URL` set to `""` (empty = same origin). A nice detail: FastAPI checks its route list on every request, so we can add a page to a server that is already running — no restart needed.

In [14]:
from fastapi.responses import HTMLResponse

# The same page, one change: BASE_URL becomes "" -> fetch("/predict") is a
# relative URL that always points at whichever server sent the page.
served_page = html_page.replace(
    'const BASE_URL = "http://127.0.0.1:8123";  // change to your Cloud Run URL to go live',
    'const BASE_URL = "";  // empty = same origin: whichever server sent this page',
)

@app.get("/", response_class=HTMLResponse)
def home():
    return served_page

# Prove it works: ask the running server for the page, like a browser would.
check = requests.get(f"{BASE_URL}/", timeout=5)
print("GET / ->", check.status_code, "| page contains the form:", "<button" in check.text)

GET / -> 200 | page contains the form: True


### Open your app — locally or on Colab

- **Locally**, the app is simply at `http://127.0.0.1:8123/` — click the link the next cell prints and use it.
- **On Colab**, Google provides a built-in **port proxy**: `google.colab.kernel.proxyPort(8123)` returns a private link from *your* browser into port 8123 of *Google's* machine. Only your Google session can open it. Same app, working inside Colab — no deployment, no configuration.

**What the next cell does / why:** it detects where the notebook is running and prints the right link. FastAPI also auto-generates an interactive documentation page for every API at `/docs` — a free bonus where you can try every endpoint by clicking.

In [15]:
import sys

if "google.colab" in sys.modules:
    # Colab's port proxy: a private link from YOUR browser into Google's machine.
    from google.colab.output import eval_js
    app_url = eval_js("google.colab.kernel.proxyPort(8123)").rstrip("/")
else:
    app_url = BASE_URL

print("While this notebook is running, your app is live at:")
print(f"  {app_url}/       <- the web form")
print(f"  {app_url}/docs   <- automatic API documentation (try it!)")

While this notebook is running, your app is live at:
  http://127.0.0.1:8123/       <- the web form
  http://127.0.0.1:8123/docs   <- automatic API documentation (try it!)


## 7. Consuming from everything else

The pattern **never changes**: URL, method, JSON. Only the spelling differs per tool.

**curl** (the command-line tool for HTTP, available on nearly every computer) — the whole call in one line:

```bash
curl -X POST http://127.0.0.1:8123/predict -H "Content-Type: application/json" -d '{"Pclass": 3, "Sex": "male", "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.9, "Embarked": "S"}'
```

**JavaScript** — you just saw it (`fetch`). **Mobile apps** (Swift on iOS, Kotlin on Android) do the identical thing with their own HTTP libraries. Even **spreadsheets** can call APIs: Google Sheets via *Apps Script*, Excel via *Power Query* — names you can look up when you need them. If a tool can speak HTTP, it can consume your model.

## 8. Consuming third-party ML APIs

Here is a secret about the industry: **most companies consume models they did not train.** Instead of building, they call:

- **OpenAI / Anthropic** — language models (chat, summarization, code)
- **Google Vision** — image recognition
- **Translation and speech APIs** — text-to-speech, speech-to-text, translation

The HTTP pattern is *exactly* what you did today — URL, method, JSON payload, JSON response — **plus one new ingredient: an API key.**

An **API key** is a password for software: a long random string sent with each request so the provider knows *who* is calling (identification) and *whom to charge* (billing). Typical pricing, one sentence each: **per call** — every request costs a fixed fraction of a cent; **per token** — language models charge by the amount of text processed (a token is roughly three-quarters of a word).

### The iron rules of API keys

- Keys live in **environment variables on the SERVER** — never hard-coded anywhere.
- **Never in client code**: anyone can read a webpage's JavaScript with a right-click — a key there is public within hours.
- **Never in notebooks, never in git**: both get shared, and keys in git history stay there forever.

That is why real apps put their *own backend* between users and the third-party API:

```
[your app]  --->  [your backend + key]  --->  [third-party model API]
 (browser,          (your server; the           (OpenAI, Google, ...)
  mobile...)         key lives here, in an
                     environment variable)
```

The user's device never sees the key; your backend adds it server-side. This section is concept-only on purpose — no keys, no calls — but the day you get your first key, you already know everything else.

## 9. Production concerns for consumers

Four things every app that consumes a model should think about. Three are runnable right now.

### 9a. Latency: how long does a prediction take?

**Latency** is the time from sending the request to receiving the answer. Users feel it directly, so measure it — do not guess.

**What the next cell does / why:** it times 20 real prediction calls with `time.perf_counter()` and prints mean, min, and max in milliseconds.

In [16]:
import numpy as np

timings = []
for _ in range(20):
    start = time.perf_counter()
    requests.post(f"{BASE_URL}/predict", json=passenger, timeout=5)
    timings.append((time.perf_counter() - start) * 1000)   # milliseconds

print(f"Latency over 20 calls: mean {np.mean(timings):.1f} ms, "
      f"min {np.min(timings):.1f} ms, max {np.max(timings):.1f} ms")

Latency over 20 calls: mean 9.6 ms, min 8.8 ms, max 10.4 ms


(Localhost is fast. A real cloud API adds network travel time — often 50 to 300 ms. Same measurement code, though: just change `BASE_URL`.)

### 9b. Caching: do not ask the same question twice

If your app asks for the *same* passenger repeatedly, why pay the round trip every time? A **cache** remembers previous answers. The model gives the same output for the same input, so the remembered answer is still correct.

**What the next cell does / why:** a tiny dict-based cache — the input (as a JSON string) is the key, the API response is the value — plus a hit counter to prove it works.

In [17]:
cache = {}
cache_hits = 0

def cached_prediction(p: dict) -> dict:
    global cache_hits
    key = json.dumps(p, sort_keys=True)        # same passenger -> same key
    if key in cache:
        cache_hits += 1
        return cache[key]                      # no HTTP call at all
    result = requests.post(f"{BASE_URL}/predict", json=p, timeout=5).json()
    cache[key] = result
    return result

for _ in range(5):
    cached_prediction(passenger)               # same passenger, five times

print(f"5 requests -> {5 - cache_hits} real API call(s), {cache_hits} served from cache")

5 requests -> 1 real API call(s), 4 served from cache


### 9c. Graceful degradation: what if the API is down?

Ask yourself early: **if the model API is down, what does your app show?** A crash and a blank screen — or a calm message and a sensible default? Handling the failure is often 3 lines.

**What the next cell does / why:** it calls a dead address on purpose and shows the fallback the user would see instead of a crash.

In [18]:
try:
    result = requests.post("http://127.0.0.1:9999/predict", json=passenger, timeout=1).json()
    message = f"Survival probability: {result['probability']:.0%}"
except requests.exceptions.RequestException:
    message = "Prediction temporarily unavailable - please try again in a moment."

print("What the user sees:", message)

What the user sees: Prediction temporarily unavailable - please try again in a moment.


### 9d. API versioning (one paragraph)

One day you will retrain the model or change what `/predict` expects — but apps written against the old version are still out there. The standard solution is a version in the URL: the API serves `/v1/predict` forever, and breaking changes go to `/v2/predict`. Old apps keep working; new apps opt in when ready. When you consume an API, prefer versioned URLs — they are a promise that the ground will not move under your feet.

## 10. Shutting down the server

**What the next cell does / why:** it asks the server to exit and waits for its thread to finish. Cleaning up connections and processes when you are done is also a professional habit — servers left running hold onto ports and memory.

In [19]:
server.should_exit = True
thread.join(timeout=10)
print("Server stopped")

Server stopped


## 11. You made it

This was the last notebook of the course. Look at the distance you covered. In **M1** you learned to clean data, describe it, and visualize it — turning raw tables into understanding. In **M2** you prepared features, selected and tuned models, and shipped your first working demo. In **M3** you learned how ML systems are architected, deployed a real prediction API to the cloud, and today closed the loop: you consumed that model the way real software does. That is the full journey — from a messy CSV to a model that any application on the internet can use. Not a toy version of it: the actual workflow, the actual tools.

**The skill map you now own:**

- **M1 — Descriptive analytics:** loading and cleaning data with pandas, summary statistics, visualization, telling honest stories with charts
- **M2 — Machine learning:** feature engineering, train/test discipline, model selection and tuning, shipping a demo
- **M3 — Architectures and deployment:** how ML systems fit together, building a FastAPI prediction service, deploying to Cloud Run / App Runner, consuming models over HTTP like a professional

**Where to go next:** build something of your own. Pick a dataset you actually care about — your sport, your music, your city — and walk it through the same pipeline: explore (M1), model (M2), deploy and consume (M3). The patterns in these notebooks are your template; only the data changes. That first self-chosen project teaches more than any course can. Good luck — you are ready.

## Key takeaways

- A deployed model is **just a URL**. Every app — web, mobile, backend, spreadsheet — consumes it the same way: HTTP request with JSON in, JSON back.
- Every call has four ingredients: **URL, method, payload, headers**. GET reads; POST sends data.
- Status codes: **2xx success, 4xx your fault, 5xx their fault**. Read the error body — the API tells you what is wrong.
- Good clients always set a **timeout**, handle errors with `raise_for_status()`, and **retry with backoff** — but only for problems that can fix themselves (503, hiccups), never for bad input (400/422).
- Hide the details in one **reusable client function**; that is what production client code looks like.
- **Batch endpoints** turn many round trips into one: fewer round trips = faster.
- **CORS** is a browser safety rule: pages may only call servers that say it is OK — that is why the API carries `CORSMiddleware`.
- An API can **serve its own page**: same origin, relative `fetch("/predict")`, no CORS and no configured address — the app works wherever the API runs, even inside Colab.
- Third-party ML APIs are the same HTTP pattern plus an **API key** — which lives in a server-side environment variable, never in client code, notebooks, or git.
- Consumers should think about **latency, caching, graceful degradation, and versioning** — small habits, big difference.

## Practice exercises

Try these while the server is still running — re-run the cells of section 2 if you already executed the shutdown cell.

1. **A client with proof.** Write a function `robust_health(url)` that tries `GET {url}/health` at most 3 times with backoff and returns the JSON on success or `None` after 3 failures. Prove both behaviors: call it once against `BASE_URL` and once against `http://127.0.0.1:9999`.
2. **Batch vs singles, measured.** Using `time.perf_counter()`, measure the latency of one `/predict-batch` call with 10 passengers vs 10 single `/predict` calls. Print both times and the speedup.
3. **Extend the web page.** Add `SibSp` and `Parch` number inputs to `outputs/webapp/index.html` and use their values in the payload instead of the hard-coded zeros.
4. **API keys, in 3 sentences.** A mobile app uses OpenAI to summarize notes. Explain in 3 sentences where the API key lives and why it must not live in the app itself.

## Solutions

The code solutions are shown as text (not runnable cells) because the server was shut down above. To run them, re-execute the server cells in section 2 first.

### Solution 1: a client with proof

```python
def robust_health(url, attempts=3, timeout=2):
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(f"{url}/health", timeout=timeout)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as err:
            print(f"attempt {attempt} failed: {type(err).__name__}")
            if attempt == attempts:
                return None
            time.sleep(0.3 * attempt)   # backoff: 0.3 s, 0.6 s

print(robust_health(BASE_URL))                  # {'status': 'ok'} on the first try
print(robust_health("http://127.0.0.1:9999"))   # three failed attempts, then None
```

The first call succeeds immediately; the second prints three failure lines and returns `None` instead of crashing — exactly the behavior a production client needs.

### Solution 2: batch vs singles, measured

```python
ten = (passengers * 2)[:10]

start = time.perf_counter()
for p in ten:
    requests.post(f"{BASE_URL}/predict", json=p, timeout=5)
singles_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
requests.post(f"{BASE_URL}/predict-batch", json=ten, timeout=5)
batch_ms = (time.perf_counter() - start) * 1000

print(f"10 single calls: {singles_ms:.1f} ms")
print(f"1 batch call:    {batch_ms:.1f} ms")
print(f"Speedup: {singles_ms / batch_ms:.1f}x")
```

Expect the batch call to be several times faster even on localhost; over a real network the gap grows with every millisecond of round-trip time.

### Solution 3: extend the web page

Add two labels to the form, next to the Age input:

```html
<label>Siblings/spouses aboard <input id="sibsp" type="number" value="0" min="0"></label>
<label>Parents/children aboard <input id="parch" type="number" value="0" min="0"></label>
```

Then replace the hard-coded zeros in the JavaScript payload:

```javascript
SibSp: Number(document.getElementById("sibsp").value),
Parch: Number(document.getElementById("parch").value),
```

Refresh the page in the browser (with the server running) and the new fields flow into the prediction. Notice what did *not* change: the URL, the method, the response handling. Adding a field to the payload is all it took.

### Solution 4: API keys, in 3 sentences

The API key lives in an environment variable on the company's own backend server, and the mobile app talks only to that backend, which adds the key before forwarding requests to OpenAI. The key cannot live in the app itself because anyone can download the app and extract every string inside it, making the key public. Whoever holds the key can spend the company's money on OpenAI's bill, so it must stay where users can never see it: on the server.